In [ ]:
import json
with open("/content/jewish_concert_archive.json","r") as f:
  info=json.load(f)
events=info["concerts"]


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = input("OpenAI key: ")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
llm = init_chat_model("gpt-4o-mini", model_provider='openai')
prompt="You are a musicologist analyzing a JSON file containing information about Jewish-Australian concert programs during wartime and internment. "+input()+" The following is the JSON data provided as a string. Use no other information in your response: "+str(events)
response = llm.invoke(prompt)
print(response.content)


In [ ]:
import plotly.graph_objects as go
import networkx as nx
import matplotlib.pyplot as plt
import pyvis
from pyvis import network as net

people=nx.Graph()
for i in range(len(events)):

  if events[i]["location"]=="Hay Internment Camp, New South Wales, Australia":
    c="blue"
  elif events[i]["location"]=="Tatura Internment Camp, Victoria, Australia":
    c="green"
  else:
    c="purple"
  people.add_node(events[i]["title"],color=c,size=10)

elabels={}
for i in range(len(events)):
  for j in range(0,i):
    namelist=[]
    for k in events[i]["Complete credits"]:
      if k in events[j]["Complete credits"]:
        namelist.append(k)
    if len(namelist)>0:
      people.add_edge(events[i]["title"],events[j]["title"], color="white",title=", ".join(namelist), font_size=8)
        #elabels.update({(events[i]["title"],events[j]["title"]):k})
#pos = nx.spring_layout(people, k=0.6, iterations=20)
#nx.draw(people,pos,node_size=50,labels={node: node for node in people.nodes()},font_size=5,font_color="blue")
#nx.draw_networkx_edge_labels(people,pos,edge_labels=elabels,font_size=5)
s_by_a={}
for i in range(len(events)):
  s_by_a.update({events[i]["title"]:[]})
  if "acts" in events[i].keys():
    for j in events[i]["acts"]:
      for k in j["songs"]:
        s_by_a[events[i]["title"]].append(k["title"])
print(s_by_a)
todo=[]
for i in s_by_a.keys():
  for j in todo:
    songs=[]
    for k in s_by_a[i]:
      if k in s_by_a[j]:
        #print(k)
        #print(i)
        #print(j)
        #print(s_by_a[j].index(k))
        songs.append(k)
    if len(songs)>0:
      people.add_edge(i,j, color="yellow",title=", ".join(songs), font_size=8)
  todo.append(i)
pyvis_graph = net.Network(notebook=True,
                          width="1000",
                          height="700",
                          bgcolor="black",
                          font_color="blue")

pyvis_graph.from_nx(people)
pyvis_graph.options.edges.smooth.enabled = True
for n in pyvis_graph.nodes:
    n["font"]={"size":9,"color":n["color"]}
pyvis_graph.show('my_new_graph.html')



In [ ]:
import pandas as pd
import seaborn as sbs
settings={"Hay Internment Camp, New South Wales, Australia":{},"Tatura Internment Camp, Victoria, Australia":{},"Melbourne, Victoria, Australia":{},"Other":{}}
for i in settings.keys():
  for j in ["1939","1940","1941","1942",'1943',"1944","Not specified"]:
    settings[i].update({j:0})
for i in range(len(events)):
  if events[i]["location"] not in ["Hay Internment Camp, New South Wales, Australia", "Tatura Internment Camp, Victoria, Australia", "Melbourne, Victoria, Australia"]:
    settings["Other"][events[i]["date"]]+=1
  else:
    settings[events[i]["location"]][events[i]["date"]]+=1
print(settings)
sbs.heatmap(pd.DataFrame(settings),annot=True,cbar=False)


